In [0]:
%sql

select * from user_behaviour_catalog.user_behaviour_schema.ecomm_behaviour limit 10 

In [0]:
df = spark.table("user_behaviour_catalog.user_behaviour_schema.ecomm_behaviour")
df.display()

# Schema and row count
df.printSchema()
df.count()

Missing Value Check

In [0]:
from pyspark.sql.functions import col, sum, when

df.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns]).display()


Event Type Distribution

In [0]:
df.groupBy("event_type").count().orderBy("count", ascending=False).display()


Top Categories / Brands

In [0]:
df.groupBy("category_code").count().orderBy("count", ascending=False).display()
df.groupBy("brand").count().orderBy("count", ascending=False).display()


Price Distribution

In [0]:
df.select("price").summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max").display()


Time Series Preparation (if event_time is string)

In [0]:
from pyspark.sql.functions import to_timestamp

df = df.withColumn("event_timestamp", to_timestamp("event_time"))
df.select("event_timestamp").orderBy("event_timestamp").display()


In [0]:
%sql
SELECT DISTINCT DATE(event_time) AS event_date
FROM user_behaviour_catalog.user_behaviour_schema.ecomm_behaviour
ORDER BY event_date;

In [0]:
event_dist = df.groupBy("event_type").count()
display(event_dist.orderBy("count", ascending=False))

 Step-by-Step EDA with Visualizations in Databricks

In [0]:
df_spark = spark.table("user_behaviour_catalog.user_behaviour_schema.ecomm_behaviour")

from pyspark.sql.functions import to_timestamp, to_date
df_spark = df_spark.withColumn("event_timestamp", to_timestamp("event_time"))
df_spark = df_spark.withColumn("event_date", to_date("event_timestamp"))

# Convert to Pandas (use only if data is not too large)
df = df_spark.toPandas()


In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set(style="whitegrid")


Event Type Distribution

In [0]:
plt.figure(figsize=(8,5))
sns.countplot(data=df, x='event_type', order=df['event_type'].value_counts().index, palette='Set2')
plt.title("Event Type Distribution")
plt.xlabel("Event Type")
plt.ylabel("Count")
plt.show()


Top Categories

In [0]:
plt.figure(figsize=(10,5))
top_categories = df['category_code'].value_counts().nlargest(10)
sns.barplot(x=top_categories.values, y=top_categories.index, palette="coolwarm")
plt.title("Top 10 Categories")
plt.xlabel("Count")
plt.ylabel("Category")
plt.show()


Time Series Trend

In [0]:
ts_data = df.groupby(['event_date', 'event_type']).size().reset_index(name='count')

fig = px.line(ts_data, x='event_date', y='count', color='event_type',
              title="Event Trends Over Time")
fig.show()


Price Distribution

In [0]:
plt.figure(figsize=(10,5))
sns.histplot(df['price'], bins=30, kde=True, color="skyblue")
plt.title("Price Distribution")
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.show()


Session Lengths

In [0]:
session_counts = df.groupby("user_session").size()

plt.figure(figsize=(10,5))
sns.histplot(session_counts, bins=30, color="purple")
plt.title("User Session Length Distribution")
plt.xlabel("Number of Events per Session")
plt.ylabel("Frequency")
plt.show()


Missing Values Heatmap

In [0]:
plt.figure(figsize=(10,5))
sns.heatmap(df.isnull(), cbar=False, cmap='Reds')
plt.title("Missing Values Heatmap")
plt.show()

Cardinality Check for ML

In [0]:
print("Unique Users:", df['user_id'].nunique())
print("Unique Products:", df['product_id'].nunique())
print("Unique Brands:", df['brand'].nunique())
print("Unique Categories:", df['category_code'].nunique())
